In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FlipkartPipeline").getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

bronze_df = spark.createDataFrame(data, columns)

In [0]:
bronze_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze_orders")

In [0]:
bronze_df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
silver_df = spark.read.table("bronze_orders")

In [0]:
from pyspark.sql.functions import col, to_date

silver_df = silver_df \
    .withColumn("amount", col("amount").cast("int")) \
    .withColumn("date", to_date(col("date")))

In [0]:
from pyspark.sql.functions import when

silver_df = silver_df \
    .withColumn("amount", when(col("amount").isNull(), 0).otherwise(col("amount"))) \
    .withColumn("city", when(col("city").isNull(), "Unknown").otherwise(col("city")))

In [0]:
silver_df = silver_df.filter(col("amount") >= 0)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

silver_df = silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

In [0]:
silver_df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|     0|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|  Unknown|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
gold_df = spark.read.table("silver_orders")

In [0]:
from pyspark.sql.functions import sum

sales_by_product = gold_df.groupBy("product") \
    .agg(sum("amount").alias("total_sales"))

In [0]:
sales_by_category = gold_df.groupBy("category") \
    .agg(sum("amount").alias("total_sales"))

In [0]:
sales_by_city = gold_df.groupBy("city") \
    .agg(sum("amount").alias("total_sales"))

In [0]:
orders_per_customer = gold_df.groupBy("customer_id").count()

In [0]:
customer_spending = gold_df.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spending"))

In [0]:
top_product = sales_by_product.orderBy(col("total_sales").desc())
top_customer = customer_spending.orderBy(col("total_spending").desc())

In [0]:
sales_by_product.write.format("delta").mode("overwrite").saveAsTable("gold_sales_product")

sales_by_category.write.format("delta").mode("overwrite").saveAsTable("gold_sales_category")

sales_by_city.write.format("delta").mode("overwrite").saveAsTable("gold_sales_city")

customer_spending.write.format("delta").mode("overwrite").saveAsTable("gold_customer_spending")

In [0]:
spark.sql("SELECT * FROM bronze_orders").show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
spark.sql("SELECT * FROM silver_orders").show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|     0|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|  Unknown|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
spark.sql("SELECT * FROM gold_sales_product").show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|    Tablet|      22000|
|    Mobile|      33000|
|     Watch|       8000|
|    Laptop|     105000|
|Headphones|       3000|
+----------+-----------+

